In [1]:
# ==========================================
# 증강 실험
# ==========================================
!pip install -q trl transformers datasets accelerate peft bitsandbytes pyarrow --upgrade

from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 9.6 MB/s eta 0:00:00
Mounted at /content/drive


In [2]:
import torch, random, pandas as pd, numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from datasets import Dataset, concatenate_datasets
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

In [3]:
# ── 1. 데이터 로드 ──────────────────────────
train = pd.read_csv('/content/drive/MyDrive/시냅스 팀플_2/dataset/train.csv')
test  = pd.read_csv('/content/drive/MyDrive/시냅스 팀플_2/dataset/test.csv')
train_data, val_data = train_test_split(train, test_size=0.1, random_state=42)
print(f"✅ 데이터 로드 | 학습: {len(train_data)}개 | 검증: {len(val_data)}개")

✅ 데이터 로드 | 학습: 10136개 | 검증: 1127개


In [4]:
# ── 2. 모델 & 토크나이저 로드 ────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)
MODEL_ID  = 'beomi/gemma-ko-2b'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'right'
print("✅ 토크나이저 로드 완료")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.11k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/555 [00:00<?, ?B/s]

✅ 토크나이저 로드 완료


In [5]:
# ── 3. 프롬프트 함수 (baseline과 동일) ───────
def create_prompt(input_text, output_text=''):
    system_prompt = (
        "당신은 난독화된 한국어 리뷰를 자연스럽고 정확한 문장으로 복원하는 전문가이다.\n"
        "반드시 복원된 문장만 출력하라.\n\n"
        "Example 1:\nInput: 배쏭 개빠름 진짜 죠아용\nOutput: 배송 개빠름 진짜 좋아용\n\n"
        "Example 2:\nInput: 마싯구 양 많아여!!\nOutput: 맛있고 양 많아요!!\n\n"
    )
    prompt = f"{system_prompt}Input: {input_text}\nOutput: "
    if output_text:
        prompt += output_text
    return prompt

In [6]:
# ── 4. F1 계산 함수 ──────────────────────────
def char_f1(pred, answer):
    pred_chars = Counter(pred)
    ans_chars  = Counter(answer)
    common = sum((pred_chars & ans_chars).values())
    if common == 0:
        return 0.0
    precision = common / sum(pred_chars.values())
    recall    = common / sum(ans_chars.values())
    return 2 * precision * recall / (precision + recall)

In [7]:
# ── 5. 증강 함수 ─────────────────────────────
CHOSUNG_LIST = ['ㄱ','ㄲ','ㄴ','ㄷ','ㄸ','ㄹ','ㅁ','ㅂ','ㅃ','ㅅ','ㅆ','ㅇ','ㅈ','ㅉ','ㅊ','ㅋ','ㅌ','ㅍ','ㅎ']

def get_chosung(c):
    return CHOSUNG_LIST[(ord(c)-0xAC00)//(21*28)] if '가'<=c<='힣' else c

def aug_random_spacing(text, prob=0.3):
    result = []
    for c in text:
        result.append(c)
        if c != ' ' and random.random() < prob:
            result.append(' ')
    return ''.join(result).strip()

def aug_chosung(text, prob=0.3):
    return ''.join(get_chosung(c) if '가'<=c<='힣' and random.random()<prob else c for c in text)

PHONETIC_MAP = {'좋':'조','없':'업','있':'읻','같':'갓','많':'만','않':'안',
                '맞':'맛','해':'헤','의':'에','되':'돼','뭐':'머','했':'헷','겠':'겟'}

def aug_phonetic(text, prob=0.4):
    return ''.join(PHONETIC_MAP[c] if c in PHONETIC_MAP and random.random()<prob else c for c in text)

def aug_special_char(text, prob=0.15):
    SPECIAL = ['@','#','*','~','!','?','ㅠ','ㅜ']
    result = []
    for c in text:
        result.append(c)
        if c != ' ' and random.random() < prob:
            result.append(random.choice(SPECIAL))
    return ''.join(result)

def aug_repeat_char(text, prob=0.15):
    REPEAT = ['ㅋ','ㅎ','ㅠ','!','ㅜ']
    result = []
    for c in text:
        result.append(c)
        if '가'<=c<='힣' and random.random()<prob:
            result.append(random.choice(REPEAT)*random.randint(1,3))
    return ''.join(result)

ALL_METHODS = [aug_random_spacing, aug_chosung, aug_phonetic, aug_special_char, aug_repeat_char]

def augment_text(text):
    selected = random.sample(ALL_METHODS, k=random.randint(1, 3))
    for method in selected:
        text = method(text)
    return text

In [8]:
# ── 6. 증강 데이터셋 생성 ────────────────────
def create_augmented_df(df, aug_multiplier=2):
    rows = []
    for _, row in df.iterrows():
        original = row['output']
        for _ in range(aug_multiplier):
            rows.append({'input': augment_text(original), 'output': original})
    return pd.DataFrame(rows)

print("증강 중...")
aug_df       = create_augmented_df(train_data, aug_multiplier=2)
combined_df  = pd.concat([train_data, aug_df], ignore_index=True).sample(frac=1, random_state=42)
print(f"✅ 원본: {len(train_data)}개 | 증강: {len(aug_df)}개 | 합계: {len(combined_df)}개")

증강 중...
✅ 원본: 10136개 | 증강: 20272개 | 합계: 30408개


In [9]:
# ── 7. 데이터셋 포맷 변환 ────────────────────
def format_row(row):
    prompt     = create_prompt(row['input'], row['output'])
    row['text']      = prompt
    row['input_ids'] = tokenizer.encode(prompt, truncation=True, max_length=256)
    return row

aug_train_dataset = Dataset.from_pandas(combined_df[['input','output']], preserve_index=False)
val_dataset       = Dataset.from_pandas(val_data[['input','output']],   preserve_index=False)

aug_train_dataset = aug_train_dataset.map(format_row, batched=False)
val_dataset       = val_dataset.map(format_row,       batched=False)
print("✅ 토크나이징 완료")

Map:   0%|          | 0/30408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1127 [00:00<?, ? examples/s]

✅ 토크나이징 완료


In [10]:
# ── 8. 모델 로드 & LoRA 적용 ─────────────────
model_aug = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config,
    device_map='auto', torch_dtype=torch.float16
)
lora_config = LoraConfig(
    r=8, lora_alpha=32,
    target_modules=['q_proj','v_proj'],
    lora_dropout=0.1, bias='none', task_type='CAUSAL_LM'
)
model_aug = get_peft_model(model_aug, lora_config)
print("✅ 모델 로드 완료")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/default/ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cpu/ops.py:36: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✅ 모델 로드 완료


In [ ]:
# ── 9. 학습 ──────────────────────────────────
sft_config = SFTConfig(
    output_dir='./results_aug',
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True, bf16=False,
    logging_steps=10,
    eval_strategy='steps', eval_steps=100,
    save_strategy='epoch',
    report_to='none',
    dataset_text_field='text',
)

trainer_aug = SFTTrainer(
    model=model_aug,
    train_dataset=aug_train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    args=sft_config,
)

print("🚀 증강 학습 시작!")
trainer_aug.train()
trainer_aug.model.save_pretrained('lora_adapter_aug')

/tmp/ipykernel_513/1812286282.py:2: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  sft_config = SFTConfig(
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 1}.


🚀 증강 학습 시작!


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cpu/ops.py:80: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cpu/ops.py:132: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [ ]:
# ── 10. 증강 전후 F1 비교 ────────────────────
from transformers import pipeline as hf_pipeline

# 학습된 모델로 pipeline 생성
pipe_aug = hf_pipeline(
    'text-generation',
    model=model_aug,
    tokenizer=tokenizer,
    device_map='auto'
)

def evaluate_f1(pipe, df, n=30):
    scores = []
    for _, row in df.head(n).iterrows():
        try:
            prompt = create_prompt(row['input'])
            out    = pipe(
                prompt,
                max_new_tokens=50,
                do_sample=False,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.eos_token_id,  # ← 경고 방지
                return_full_text=True
            )
            pred = out[0]['generated_text'][len(prompt):].strip()
            # 줄바꿈 이후 텍스트 제거 (불필요한 생성 방지)
            pred = pred.split('\n')[0].strip()
            scores.append(char_f1(pred, row['output']))
        except Exception as e:
            print(f"스킵: {e}")
            scores.append(0.0)
    return sum(scores) / len(scores) if scores else 0.0

f1_baseline = 0.1409
print("결과 측정 중... (30개 샘플)")
f1_aug = evaluate_f1(pipe_aug, val_data)

print("\n========== 데이터 증강 실험 결과 ==========")
print(f"증강 없음 (baseline):  Char F1 = {f1_baseline:.4f}")
print(f"증강 있음:             Char F1 = {f1_aug:.4f}")
print(f"성능 변화:             {'+' if f1_aug > f1_baseline else ''}{f1_aug - f1_baseline:.4f}")